# Preparación de vectores de concentración de PM10 y PM2.5

Cada observación reúne dos mediciones de concentración en el orden **[PM10, PM2.5]**, expresadas en µg/m³. Se utiliza exclusivamente la salida del cuaderno de limpieza, identificada mediante su manifiesto.

La coincidencia se determina por informe, evaluación, identidad espacial, fecha, hora, periodo y la información declarada del final de medición. No se promedian concentraciones, no se imputan contaminantes ausentes y no se replican filas. Se conserva la cobertura completa de la limpieza, sin seleccionar años por antigüedad.

Las mediciones de una hora y de 24 horas se mantienen identificadas y se exportan también por separado. Las coordenadas y los códigos de estación son metadatos; no se incorporan como dimensiones numéricas del vector.

## 1. Entorno y origen de los datos

La entrada se obtiene de `Limpieza_PM/ultima_ejecucion.json`. Se verifica que corresponda a una limpieza completada y que el conteo del Parquet coincida con su manifiesto. DuckDB realiza el pivotado sin cargar todas las mediciones como un DataFrame.

In [1]:
%pip -q install duckdb

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pathlib import Path
from datetime import datetime,timezone
import json,hashlib,os
import duckdb
import pandas as pd
from IPython.display import display,Markdown

DATA_DIR=Path('/content/drive/MyDrive/Concurrente')
MANIFEST_PATH=DATA_DIR/'Limpieza_PM'/'ultima_ejecucion.json'
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError('No se encontró el manifiesto de limpieza. Ejecuta primero Limpieza.ipynb.')
manifest_bytes=MANIFEST_PATH.read_bytes()
origin=json.loads(manifest_bytes)
if origin.get('etapa')!='limpieza' or origin.get('estado')!='completado':
    raise ValueError('La entrada debe corresponder a una limpieza completada.')
SOURCE=Path(origin['parquet'])
if not SOURCE.is_file():raise FileNotFoundError('No se encontró el Parquet indicado por la limpieza.')
EXPECTED_KEYS=['informe','evaluacion','punto','este','norte','zona','datum','fecha','hora','periodo','fecha_fin_clave','hora_fin_clave']
if origin.get('clave_observacion')!=EXPECTED_KEYS:
    raise ValueError('La clave del manifiesto cambió; revisa su definición antes de preparar vectores.')
keys=list(origin['clave_observacion'])
features=['PM10','PM2.5']
RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
OUT=DATA_DIR/'Vectores_PM'/RUN_ID
OUT.mkdir(parents=True,exist_ok=False)
con=duckdb.connect()
con.execute("SET memory_limit='2GB'")
con.execute(f'SET threads={min(os.cpu_count() or 2,4)}')
def sqlpath(p):return str(p).replace("'","''")
def qi(s):return '"'+s.replace('"','""')+'"'
key_sql=','.join(qi(k) for k in keys)
tables={}
def report(name,sql):
    frame=con.execute(sql).df();tables[name]=frame
    frame.to_csv(OUT/(name+'.csv'),index=False)
    display(Markdown('**'+name+'**'));display(frame.head(50))
    if len(frame)>50:print(f'Se muestran 50 de {len(frame):,} filas; el CSV conserva la tabla completa.')
    return frame
con.execute("CREATE VIEW mediciones AS SELECT * FROM read_parquet('"+sqlpath(SOURCE)+"')")
columns=set(con.execute('DESCRIBE mediciones').df().column_name)
required=set(keys+['contaminante','concentracion','unidad','datum_reconocido'])
if required-columns:raise ValueError('Faltan columnas: '+str(sorted(required-columns)))
n_input=int(con.execute('SELECT count(*) FROM mediciones').fetchone()[0])
if n_input!=int(origin['filas']):raise ValueError('El conteo no coincide con el manifiesto de limpieza.')
print('Ejecución de limpieza:',origin['run_id'])
print('Mediciones reales de entrada:',f'{n_input:,}')
display(con.execute('SELECT punto,fecha,hora,periodo,contaminante,concentracion,unidad FROM mediciones LIMIT 5').df())

Ejecución de limpieza: 20261105T003520_811897Z
Mediciones reales de entrada: 1,062,897


,punto,fecha,hora,periodo,contaminante,concentracion,unidad
0,CA-SOC-01,2024-04-15,17:00:00,1 hora,PM2.5,15.77,µg/m³
1,CA-SOC-01,2024-04-16,04:00:00,1 hora,PM2.5,30.09,µg/m³
2,CA-SOC-01,2024-04-16,07:00:00,1 hora,PM2.5,36.60,µg/m³
3,CA-SOC-01,2024-04-16,20:00:00,1 hora,PM2.5,48.28,µg/m³
4,CA-SOC-01,2024-04-17,02:00:00,1 hora,PM10,49.18,µg/m³


## 2. Identidad de la observación y validaciones

La clave de observación se reutiliza sin modificar la definición adoptada en la limpieza. `fecha_fin_clave` y `hora_fin_clave` conservan el final normalizado o la representación original cuando no se pudo interpretar. Un marcador de final desconocido no se convierte en una fecha; permite mantener explícita la información declarada del registro.

No se exige un final completo a todas las lecturas: en muchas mediciones horarias no está informado. La coincidencia exige el mismo inicio y periodo, además del resto de la clave. Si los finales declarados difieren, no se fuerza su unión.

Antes de pivotar se comprueban valores finitos no negativos, unidad homogénea, componentes, claves y unicidad por contaminante. Una repetición inesperada detiene el procedimiento para revisar la limpieza; no se oculta mediante un promedio.

In [4]:
core_keys=[k for k in keys if k not in ['fecha_fin_clave','hora_fin_clave']]
missing_condition=' OR '.join(qi(k)+' IS NULL' for k in core_keys)
missing_condition+=' OR fecha_fin_clave IS NULL OR hora_fin_clave IS NULL'
invalid=int(con.execute("""SELECT count(*) FROM mediciones WHERE contaminante IS NULL
 OR contaminante NOT IN ('PM10','PM2.5') OR concentracion IS NULL OR NOT isfinite(concentracion)
 OR concentracion<0 OR unidad IS NULL OR unidad<>'µg/m³' OR periodo IS NULL
 OR periodo NOT IN ('1 hora','24 horas') OR datum_reconocido IS NULL OR ("""+missing_condition+')').fetchone()[0])
if invalid:raise ValueError(f'Hay {invalid:,} mediciones que incumplen el contrato de entrada. Revisa la limpieza.')
con.execute('CREATE TABLE conteos AS SELECT '+key_sql+',contaminante,count(*) AS n FROM mediciones GROUP BY ALL')
n_repeated=int(con.execute('SELECT count(*) FROM conteos WHERE n>1').fetchone()[0])
if n_repeated:
    report('ERROR_claves_repetidas','SELECT * FROM conteos WHERE n>1 ORDER BY n DESC')
    raise ValueError('Se encontraron claves repetidas por contaminante. No se aplicará un promedio.')
print('Mediciones validadas:',f'{n_input:,}')
print('Claves repetidas por contaminante:',n_repeated)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Mediciones validadas: 1,062,897
Claves repetidas por contaminante: 0


## 3. Construcción de pares completos

Se transforman las mediciones a columnas PM10 y PM2.5. La selección de un valor por columna utiliza la unicidad comprobada en la sección anterior; no se trata de una agregación de concentraciones diferentes.

Una clave que contiene solo un contaminante queda fuera de los vectores y se conserva en un archivo de observaciones incompletas. No se rellena el componente faltante. El identificador de vector se asigna después de ordenar por la clave; sirve para enlazar matriz y metadatos dentro de esta ejecución.

In [5]:
con.execute("""CREATE TABLE observaciones AS SELECT """+key_sql+""",
 max(concentracion) FILTER(WHERE contaminante='PM10') AS "PM10",
 max(concentracion) FILTER(WHERE contaminante='PM2.5') AS "PM2.5",
 bool_and(datum_reconocido) AS datum_reconocido
 FROM mediciones GROUP BY """+key_sql)
con.execute('CREATE VIEW incompletas AS SELECT * FROM observaciones WHERE "PM10" IS NULL OR "PM2.5" IS NULL')
con.execute('CREATE TABLE vectores AS SELECT row_number() OVER(ORDER BY '+key_sql+') AS vector_id,* FROM observaciones WHERE "PM10" IS NOT NULL AND "PM2.5" IS NOT NULL')
n_pairs=int(con.execute('SELECT count(*) FROM vectores').fetchone()[0])
n_incomplete=int(con.execute('SELECT count(*) FROM incompletas').fetchone()[0])
assert 2*n_pairs+n_incomplete==n_input
assert con.execute('SELECT count(*) FROM vectores WHERE NOT isfinite("PM10") OR NOT isfinite("PM2.5") OR "PM10"<0 OR "PM2.5"<0').fetchone()[0]==0
report('01_balance_vectores',f"""SELECT 'Mediciones limpias de entrada' AS etapa,{n_input}::BIGINT AS cantidad
 UNION ALL SELECT 'Mediciones utilizadas en pares completos',{2*n_pairs}
 UNION ALL SELECT 'Mediciones sin el otro contaminante',{n_incomplete}
 UNION ALL SELECT 'Vectores completos [PM10, PM2.5]',{n_pairs}
 UNION ALL SELECT 'Claves repetidas por contaminante',{n_repeated}""")
display(con.execute('SELECT vector_id,punto,fecha,hora,periodo,"PM10","PM2.5",datum,datum_reconocido FROM vectores ORDER BY vector_id LIMIT 10').df())
print('Pares reales PM10/PM2.5:',f'{n_pairs:,}')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

**01_balance_vectores**

,etapa,cantidad
0,Mediciones limpias de entrada,1062897
1,Mediciones utilizadas en pares completos,927288
2,Mediciones sin el otro contaminante,135609
3,"Vectores completos [PM10, PM2.5]",463644
4,Claves repetidas por contaminante,0


,vector_id,punto,fecha,hora,periodo,PM10,PM2.5,datum,datum_reconocido
0,1,CA-01,2019-05-07,09:05:00,24 horas,31.0,12.0,WGS84,True
1,2,CA-01,2019-05-08,09:40:00,24 horas,26.0,11.0,WGS84,True
2,3,CA-01,2019-05-09,10:00:00,24 horas,37.0,17.0,WGS84,True
3,4,CA-01,2019-05-15,11:48:00,24 horas,20.0,7.0,WGS84,True
4,5,CA-01,2019-05-16,11:55:00,24 horas,19.0,6.0,WGS84,True
5,6,CA-03,2019-04-24,00:00:00,1 hora,0.0,0.0,WGS84,True
6,7,CA-03,2019-04-24,00:00:00,24 horas,59.0,4.0,WGS84,True
7,8,CA-03,2019-04-24,01:00:00,1 hora,0.0,0.0,WGS84,True
8,9,CA-03,2019-04-24,02:00:00,1 hora,0.0,0.0,WGS84,True
9,10,CA-03,2019-04-24,03:00:00,1 hora,0.0,0.0,WGS84,True


Pares reales PM10/PM2.5: 463,644


## 4. Cobertura y distribución de los vectores

La cobertura se presenta por año y periodo. Las estadísticas de las dos dimensiones se calculan por separado para una hora y 24 horas. Estos conteos no equivalen al volumen de mediciones procesadas en la carga.

Los vectores con datum pendiente de revisión mantienen esa condición en los metadatos. Sus concentraciones pueden describirse, pero la referencia espacial no se presenta como validada ni se utiliza para calcular distancias geográficas. Un par de concentraciones no constituye por sí solo una predicción.

In [6]:
report('02_vectores_por_periodo',"""SELECT periodo,count(*) AS vectores,
 count(DISTINCT (punto,este,norte,zona,datum)) AS identidades_espaciales,
 min(fecha) AS primera_fecha,max(fecha) AS ultima_fecha,
 count(*) FILTER(WHERE NOT datum_reconocido) AS datum_pendiente_revision
 FROM vectores GROUP BY periodo ORDER BY periodo""")
report('03_vectores_por_anio',"""SELECT year(fecha) AS anio,periodo,count(*) AS vectores
 FROM vectores GROUP BY ALL ORDER BY anio,periodo""")
report('04_estadisticas_features',"""WITH valores AS (
 SELECT periodo,'PM10' AS componente,"PM10" AS concentracion FROM vectores
 UNION ALL SELECT periodo,'PM2.5',"PM2.5" FROM vectores)
 SELECT periodo,componente,count(*) AS observaciones,min(concentracion) AS minimo,
 avg(concentracion) AS promedio,median(concentracion) AS mediana,
 stddev_samp(concentracion) AS desviacion_estandar,
 quantile_cont(concentracion,0.25) AS q1,quantile_cont(concentracion,0.75) AS q3,
 quantile_cont(concentracion,0.99) AS p99,max(concentracion) AS maximo
 FROM valores GROUP BY ALL ORDER BY periodo,componente""")

**02_vectores_por_periodo**

,periodo,vectores,identidades_espaciales,primera_fecha,ultima_fecha,datum_pendiente_revision
0,1 hora,445347,39,2019-04-24,2024-12-31,52457
1,24 horas,18297,42,2019-01-25,2024-12-31,2187


**03_vectores_por_anio**

,anio,periodo,vectores
0,2019,1 hora,4344
1,2019,24 horas,271
2,2020,1 hora,3168
3,2020,24 horas,159
4,2021,1 hora,15738
5,2021,24 horas,447
6,2022,1 hora,155235
7,2022,24 horas,6416
8,2023,1 hora,119815
9,2023,24 horas,4907


**04_estadisticas_features**

,periodo,componente,observaciones,minimo,promedio,mediana,desviacion_estandar,q1,q3,p99,maximo
0,1 hora,PM10,445347,0.0,42.384171,23.45,57.842003,10.88,51.31,253.0008,3848.78
1,1 hora,PM2.5,445347,0.0,19.808008,11.31,26.951926,6.03,20.71,137.5800,1256.30
2,24 horas,PM10,18297,0.0,42.891401,27.10,43.837690,15.00,55.40,201.6120,437.10
3,24 horas,PM2.5,18297,0.0,20.031112,12.60,23.585662,7.40,21.20,117.8040,203.20


,periodo,componente,observaciones,minimo,promedio,mediana,desviacion_estandar,q1,q3,p99,maximo
0,1 hora,PM10,445347,0.0,42.384171,23.45,57.842003,10.88,51.31,253.0008,3848.78
1,1 hora,PM2.5,445347,0.0,19.808008,11.31,26.951926,6.03,20.71,137.5800,1256.30
2,24 horas,PM10,18297,0.0,42.891401,27.10,43.837690,15.00,55.40,201.6120,437.10
3,24 horas,PM2.5,18297,0.0,20.031112,12.60,23.585662,7.40,21.20,117.8040,203.20


## 5. Representación para el modelo

Las dos dimensiones se exportan en **concentraciones físicas sin escalar**, con el orden fijo PM10, PM2.5. No se incluyen zona, coordenadas, hora ni identificadores como dimensiones adicionales. El algoritmo deberá seleccionar explícitamente un periodo de medición.

El escalado pertenece a la configuración del modelo y debe justificarse. Si se aplica, sus parámetros deben conservarse y utilizarse de manera idéntica en las implementaciones comparadas. Para predicción, los parámetros se ajustan solo con entrenamiento, evitando utilizar información del periodo de evaluación. Este cuaderno no entrena K-Means ni ajusta un modelo predictivo.

## 6. Exportación y trazabilidad

Se exportan el conjunto completo con metadatos y archivos por periodo. Cada archivo de características contiene únicamente PM10 y PM2.5; su archivo de metadatos conserva el mismo orden de filas y `vector_id`. También se conserva un Parquet de claves incompletas para revisar la disponibilidad de contaminantes.

Los archivos masivos se mantienen fuera de Git. El cuaderno, los resúmenes y las tablas pequeñas permiten documentar y reproducir la preparación. El manifiesto registra la ejecución de limpieza de origen, el orden de características y los archivos producidos.

In [7]:
all_csv=OUT/'vectores_PM10_PM25.csv'
all_parquet=OUT/'vectores_PM10_PM25.parquet'
missing_parquet=OUT/'observaciones_incompletas.parquet'
def copy_query(sql,path,parquet=False):
    options='FORMAT PARQUET,COMPRESSION ZSTD' if parquet else "HEADER,DELIMITER ','"
    con.execute('COPY ('+sql+") TO '"+sqlpath(path)+"' ("+options+')')
copy_query('SELECT * FROM vectores ORDER BY vector_id',all_csv)
copy_query('SELECT * FROM vectores ORDER BY vector_id',all_parquet,True)
copy_query('SELECT * FROM incompletas ORDER BY '+key_sql,missing_parquet,True)
exports=[]
for period,tag in [('1 hora','1_hora'),('24 horas','24_horas')]:
    n=int(con.execute('SELECT count(*) FROM vectores WHERE periodo=?',[period]).fetchone()[0])
    where=" FROM vectores WHERE periodo='"+period+"' ORDER BY vector_id"
    feature_file=OUT/('features_'+tag+'.csv')
    metadata_file=OUT/('metadatos_'+tag+'.csv')
    copy_query('SELECT "PM10","PM2.5"'+where,feature_file)
    copy_query('SELECT vector_id,'+key_sql+',datum_reconocido'+where,metadata_file)
    actual=int(con.execute("SELECT count(*) FROM read_csv('"+sqlpath(feature_file)+"',header=true,all_varchar=true)").fetchone()[0])
    assert actual==n
    exports.append({'periodo':period,'filas':n,'features_csv':str(feature_file),'metadatos_csv':str(metadata_file)})
assert sum(x['filas'] for x in exports)==n_pairs
assert con.execute("SELECT count(*) FROM read_parquet('"+sqlpath(all_parquet)+"')").fetchone()[0]==n_pairs
vector_manifest={'run_id':RUN_ID,'etapa':'vectores','estado':'completado',
 'origen_limpieza_run_id':origin['run_id'],'origen_manifest_sha256':hashlib.sha256(manifest_bytes).hexdigest(),
 'mediciones_limpias':n_input,'pares_reales':n_pairs,'mediciones_sin_par':n_incomplete,
 'clave_observacion':keys,'orden_features':features,'unidad':'µg/m³','escalado':None,
 'csv':str(all_csv),'parquet':str(all_parquet),'por_periodo':exports,
 'observaciones_incompletas':str(missing_parquet),
 'cobertura':'todos los años disponibles en la limpieza'}
(OUT/'manifest.json').write_text(json.dumps(vector_manifest,indent=2,ensure_ascii=False),encoding='utf-8')
display(pd.DataFrame([{'archivo':p.name,'bytes':p.stat().st_size} for p in OUT.iterdir() if p.is_file()]))

,archivo,bytes
0,01_balance_vectores.csv,222
1,02_vectores_por_periodo.csv,182
2,03_vectores_por_anio.csv,242
3,04_estadisticas_features.csv,490
4,vectores_PM10_PM25.csv,91246186
5,vectores_PM10_PM25.parquet,2882769
6,observaciones_incompletas.parquet,301588
7,features_1_hora.csv,4845984
8,metadatos_1_hora.csv,82513961
9,features_24_horas.csv,186647


## 7. Síntesis de la preparación

La síntesis distingue mediciones individuales, claves incompletas y vectores completos. El número de vectores es el resultado de la coincidencia entre contaminantes; no se aumenta mediante copias ni se interpreta como el número de registros cargados.

In [8]:
n_unverified=int(con.execute('SELECT count(*) FROM vectores WHERE NOT datum_reconocido').fetchone()[0])
first_date,last_date=con.execute('SELECT min(fecha),max(fecha) FROM vectores').fetchone()
summary={'mediciones_limpias':n_input,'pares_reales':n_pairs,'mediciones_utilizadas_en_pares':2*n_pairs,
 'mediciones_sin_par':n_incomplete,'claves_repetidas':n_repeated,
 'orden_features':features,'vectores_por_periodo':{x['periodo']:x['filas'] for x in exports},
 'vectores_con_datum_pendiente_revision':n_unverified,
 'fecha_minima':str(first_date),'fecha_maxima':str(last_date),
 'origen_limpieza_run_id':origin['run_id'],'escalado':None}
text=f"""# Resumen de la preparación de vectores

La entrada contiene **{n_input:,} mediciones limpias**. La coincidencia por la clave documentada produjo **{n_pairs:,} vectores [PM10, PM2.5]**, que utilizan **{2*n_pairs:,} mediciones**. Las **{n_incomplete:,} mediciones restantes** no tienen el otro contaminante en la misma clave y se conservaron fuera de los pares, sin imputación.

Los vectores comprenden **{first_date} a {last_date}**, sin filtro de antigüedad. Se exportaron **{summary['vectores_por_periodo']['1 hora']:,} vectores de una hora** y **{summary['vectores_por_periodo']['24 horas']:,} de 24 horas**, en archivos separados para seleccionar el periodo del modelo.

Las concentraciones permanecen en **µg/m³**, sin escalado ni réplica. Se comprobaron **{n_repeated} claves repetidas por contaminante**. **{n_unverified:,} vectores** conservan un datum pendiente de revisión; esta condición se mantiene en sus metadatos.

La preparación no entrena el modelo ni demuestra capacidad predictiva. El número de vectores se informa separadamente del volumen de mediciones procesadas durante la limpieza.
"""
display(Markdown(text))
(OUT/'RESUMEN_VECTORES.md').write_text(text,encoding='utf-8')
(OUT/'resumen_vectores.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding='utf-8')
(DATA_DIR/'Vectores_PM'/'ultima_ejecucion.json').write_text(json.dumps(vector_manifest,indent=2,ensure_ascii=False),encoding='utf-8')
con.close()

# Resumen de la preparación de vectores

La entrada contiene **1,062,897 mediciones limpias**. La coincidencia por la clave documentada produjo **463,644 vectores [PM10, PM2.5]**, que utilizan **927,288 mediciones**. Las **135,609 mediciones restantes** no tienen el otro contaminante en la misma clave y se conservaron fuera de los pares, sin imputación.

Los vectores comprenden **2019-01-25 a 2024-12-31**, sin filtro de antigüedad. Se exportaron **445,347 vectores de una hora** y **18,297 de 24 horas**, en archivos separados para seleccionar el periodo del modelo.

Las concentraciones permanecen en **µg/m³**, sin escalado ni réplica. Se comprobaron **0 claves repetidas por contaminante**. **54,644 vectores** conservan un datum pendiente de revisión; esta condición se mantiene en sus metadatos.

La preparación no entrena el modelo ni demuestra capacidad predictiva. El número de vectores se informa separadamente del volumen de mediciones procesadas durante la limpieza.
